# 🐛 CoBug AI Learning Platform — Colab Runner

This notebook launches the CoBug AI Learning Platform inside Google Colab and gives you a public URL to access it.

---

## ✅ Before you start — get these two free keys

| Key | Where to get it | Takes |
|-----|----------------|-------|
| **Groq API Key** | [console.groq.com](https://console.groq.com) → sign up → API Keys → Create | ~1 min |
| **ngrok Auth Token** | [dashboard.ngrok.com](https://dashboard.ngrok.com) → sign up → Your Authtoken | ~1 min |

---

## 📦 Upload the project zip

1. You should have received a file called **`ai-learning-platform.zip`** alongside this notebook.
2. In Google Colab, click the **📁 folder icon** on the left sidebar.
3. Click the **⬆ Upload** button and upload `ai-learning-platform.zip`.
4. Wait for the upload to finish, then proceed.

---

## ▶ Run order

**Cell 1** → setup environment  
**Cell 2** → paste your API keys  
**Cell 3** → build and launch  

Use **Cell 4** anytime to download your data as CSV.  
Use **Cell 5** to stop the server cleanly.

> **Tip:** Use **Runtime → Run all** to execute all cells at once after filling in your keys in Cell 2.

---
## Cell 1 — Environment Setup
*Installs Node.js, extracts the project, and installs dependencies. Run once per session.*

In [ ]:
import subprocess, sys, os, zipfile, shutil, tempfile

platform_dir = '/content/ai-learning-platform'
zip_path     = '/content/ai-learning-platform.zip'

# ── Step 1: Extract project files ───────────────────────────
if os.path.exists(platform_dir):
    print('[1/5] Project already extracted, skipping.')
else:
    print('[1/5] Extracting project zip...')
    if not os.path.exists(zip_path):
        raise FileNotFoundError(
            '\n'
            'ai-learning-platform.zip not found!\n'
            'Please upload it first:\n'
            '  1. Click the 📁 folder icon in the left sidebar\n'
            '  2. Click Upload and select ai-learning-platform.zip\n'
            '  3. Wait for upload to finish, then re-run this cell.'
        )
    tmp = tempfile.mkdtemp()
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(tmp)
    # Handle both: zip contains folder inside, or files directly
    entries = os.listdir(tmp)
    subdirs  = [e for e in entries if os.path.isdir(os.path.join(tmp, e))]
    has_root_files = any(os.path.isfile(os.path.join(tmp, e)) for e in entries)
    if subdirs and not has_root_files:
        shutil.move(os.path.join(tmp, subdirs[0]), platform_dir)
    else:
        shutil.move(tmp, platform_dir)
    print('  Extracted successfully.')

# Validate expected structure
for folder in ['backend', 'frontend']:
    path = os.path.join(platform_dir, folder)
    if not os.path.isdir(path):
        raise FileNotFoundError(
            f'Expected "{folder}" folder not found inside the zip.\n'
            f'Make sure you zipped the ai-learning-platform folder itself, not its contents.'
        )
print('  Project structure OK.')

# ── Step 2: Node.js ─────────────────────────────────────────
print('[2/5] Checking Node.js...')
try:
    node_ver = subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    node_ver = ''

if not (node_ver.startswith('v18') or node_ver.startswith('v20') or node_ver.startswith('v22')):
    print('  Installing Node.js 20 (this takes ~1 minute)...')
    r = subprocess.run(
        'curl -fsSL https://deb.nodesource.com/setup_20.x | bash -',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'Node.js setup failed: {r.stderr[-400:]}')
    r = subprocess.run('apt-get install -y nodejs', shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Node.js install failed: {r.stderr[-400:]}')
    node_ver = subprocess.run(['node', '-v'], capture_output=True, text=True).stdout.strip()
print(f'  Node.js {node_ver} ready.')

# ── Step 3: pyngrok ─────────────────────────────────────────
print('[3/5] Installing pyngrok...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'pyngrok', '-q'],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f'pyngrok install failed: {r.stderr[-400:]}')
print('  pyngrok ready.')

# ── Step 4: Backend packages ─────────────────────────────────
print('[4/5] Installing backend packages...')
r = subprocess.run(
    ['npm', 'install', '--prefer-offline'],
    cwd=f'{platform_dir}/backend',
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f'Backend npm install failed:\n{r.stderr[-600:]}')
print('  Backend packages ready.')

# ── Step 5: Frontend packages ────────────────────────────────
print('[5/5] Installing frontend packages...')
r = subprocess.run(
    ['npm', 'install', '--prefer-offline'],
    cwd=f'{platform_dir}/frontend',
    capture_output=True, text=True
)
if r.returncode != 0:
    raise RuntimeError(f'Frontend npm install failed:\n{r.stderr[-600:]}')
print('  Frontend packages ready.')

print()
print('✅ Setup complete! Proceed to Cell 2.')

---
## Cell 2 — Paste Your API Keys
*Replace the placeholder values below with your real keys, then run this cell.*

In [ ]:
import os

# ============================================================
#  PASTE YOUR KEYS BELOW
# ============================================================
GROQ_API_KEY     = 'gsk_your_groq_key_here'   # Get free key at console.groq.com
NGROK_AUTH_TOKEN = 'your_ngrok_token_here'     # Get free token at dashboard.ngrok.com
GITHUB_TOKEN     = ''                          # Optional — raises GitHub API rate limit
# ============================================================

# Validate keys before saving
if not GROQ_API_KEY.strip() or GROQ_API_KEY.strip() == 'gsk_your_groq_key_here':
    raise ValueError(
        'GROQ_API_KEY is not set.\n'
        'Go to https://console.groq.com → API Keys → Create key\n'
        'Then paste it above and re-run this cell.'
    )
if not NGROK_AUTH_TOKEN.strip() or NGROK_AUTH_TOKEN.strip() == 'your_ngrok_token_here':
    raise ValueError(
        'NGROK_AUTH_TOKEN is not set.\n'
        'Go to https://dashboard.ngrok.com → Your Authtoken\n'
        'Then paste it above and re-run this cell.'
    )

# Write backend .env file
backend_dir = '/content/ai-learning-platform/backend'
os.makedirs(backend_dir, exist_ok=True)
with open(os.path.join(backend_dir, '.env'), 'w') as f:
    f.write(f'GROQ_API_KEY={GROQ_API_KEY.strip()}\n')
    f.write(f'GITHUB_TOKEN={GITHUB_TOKEN.strip()}\n')
    f.write('PORT=5000\n')

# Keep ngrok token in memory only (not written to disk)
os.environ['COBUG_NGROK_TOKEN'] = NGROK_AUTH_TOKEN.strip()

print('✅ Keys saved. Proceed to Cell 3.')

---
## Cell 3 — Build and Launch
*Builds the frontend, starts the backend, and gives you a public URL. Keep this cell running while you use the platform.*

In [ ]:
import subprocess, os, shutil, time, urllib.request, threading

platform_dir = '/content/ai-learning-platform'
frontend_dir = f'{platform_dir}/frontend'
backend_dir  = f'{platform_dir}/backend'

# Pre-flight checks
if not os.path.isdir(frontend_dir):
    raise RuntimeError('Frontend folder missing. Please run Cell 1 first.')
if not os.path.exists(f'{backend_dir}/.env'):
    raise RuntimeError('API keys not configured. Please run Cell 2 first.')
if not os.environ.get('COBUG_NGROK_TOKEN'):
    raise RuntimeError('ngrok token missing. Please run Cell 2 first.')

# ── Step 1: Build frontend ───────────────────────────────────
print('[1/4] Building frontend bundle (may take ~60 seconds)...')
build_env = os.environ.copy()
build_env['NODE_OPTIONS'] = '--max-old-space-size=512'
r = subprocess.run(
    ['npm', 'run', 'build'],
    cwd=frontend_dir,
    env=build_env,
    capture_output=True,
    text=True
)
if r.returncode != 0:
    print('❌ Frontend build failed. Output:')
    print(r.stdout[-1000:])
    print(r.stderr[-1000:])
    raise RuntimeError('Fix the build error above and re-run this cell.')
print('  Frontend built successfully.')

# ── Step 2: Copy dist → backend/public ──────────────────────
print('[2/4] Linking static files to backend server...')
public_dest = f'{backend_dir}/public'
if os.path.exists(public_dest):
    shutil.rmtree(public_dest)
shutil.copytree(f'{frontend_dir}/dist', public_dest)
print('  Static files linked.')

# ── Step 3: Start backend server ────────────────────────────
print('[3/4] Starting backend server...')
# Kill any leftover node process from a previous run
subprocess.run(['pkill', '-f', 'node'], capture_output=True)
time.sleep(1.5)

log_path = '/content/server.log'
log_file = open(log_path, 'w')
server = subprocess.Popen(
    ['node', 'server.js'],
    cwd=backend_dir,
    stdout=log_file,
    stderr=log_file
)

# Wait up to 40 seconds for the server to respond
print('  Waiting for server to start', end='', flush=True)
is_up = False
for _ in range(40):
    time.sleep(1)
    print('.', end='', flush=True)
    # Check if process crashed
    if server.poll() is not None:
        print()
        log_file.flush()
        with open(log_path) as lf:
            print('Server crashed. Log output:')
            print(lf.read())
        raise RuntimeError('Node.js server crashed. See log above.')
    try:
        with urllib.request.urlopen('http://localhost:5000/api/health', timeout=2) as resp:
            if resp.status == 200:
                print(' UP!')
                is_up = True
                break
    except Exception:
        pass

if not is_up:
    log_file.flush()
    with open(log_path) as lf:
        print('\nServer did not start in 40 seconds. Log output:')
        print(lf.read())
    raise RuntimeError('Server failed to start. See log above.')

# ── Step 4: Open ngrok tunnel ────────────────────────────────
print('[4/4] Opening public tunnel via ngrok...')
from pyngrok import ngrok, conf

conf.get_default().auth_token = os.environ['COBUG_NGROK_TOKEN']
# Close any tunnels from a previous run
try:
    ngrok.kill()
    time.sleep(1)
except Exception:
    pass

tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print()
print('=' * 62)
print('  ✅  CoBug AI Learning Platform is LIVE')
print(f'  🌐  URL: {public_url}')
print('=' * 62)
print()
print('Open the URL above in your browser.')
print('Keep this cell running — interrupting it will stop the server.')
print()

# Stream live server logs in the background
def _tail_log():
    with open(log_path, 'r') as f:
        f.seek(0, 2)  # Start from end of file
        while True:
            line = f.readline()
            if line:
                print(f'  [server] {line.rstrip()}')
            else:
                time.sleep(0.3)

threading.Thread(target=_tail_log, daemon=True).start()

# Keep cell alive
try:
    while True:
        time.sleep(10)
        # Restart server if it crashed
        if server.poll() is not None:
            print('  [server] Process ended unexpectedly, restarting...')
            log_file2 = open(log_path, 'a')
            server = subprocess.Popen(
                ['node', 'server.js'],
                cwd=backend_dir,
                stdout=log_file2,
                stderr=log_file2
            )
            print(f'  [server] Restarted (PID {server.pid})')
except KeyboardInterrupt:
    print('\nStopped. Run Cell 5 to clean up all services.')

---
## Cell 4 — Export Learning Data to CSV
*Downloads all recorded quiz and exercise results as a CSV file to your local machine.*

In [ ]:
import sqlite3, csv, os

db_path  = '/content/ai-learning-platform/backend/learning.db'
csv_path = '/content/cobug_dataset.csv'

if not os.path.exists(db_path):
    print('No database found yet.')
    print('Complete at least one quiz or exercise on the platform first.')
else:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    try:
        cursor.execute('''
            SELECT id, user_id, topic, language, skill_level,
                   score, total, time_spent, hints_used,
                   behavior_tag, recommended_action, created_at
            FROM scores
            ORDER BY created_at ASC
        ''')
        rows    = cursor.fetchall()
        columns = [d[0] for d in cursor.description]

        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(columns)
            writer.writerows(rows)

        print(f'Exported {len(rows)} records → {csv_path}')

        # Auto-download to local machine
        try:
            from google.colab import files
            files.download(csv_path)
            print('Download started in your browser.')
        except Exception:
            print(f'Download manually from the 📁 file panel: {csv_path}')

    except sqlite3.OperationalError as e:
        print(f'Database error: {e}')
    finally:
        conn.close()

---
## Cell 5 — Stop All Services
*Run this to cleanly shut down the server and close the ngrok tunnel.*

In [ ]:
import subprocess

print('Closing ngrok tunnel...')
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('  Tunnel closed.')
except Exception as e:
    print(f'  ngrok already stopped ({e})')

print('Stopping Node.js server...')
subprocess.run(['pkill', '-f', 'node'], capture_output=True)
print('  Server stopped.')
print()
print('All services stopped. You can now close this tab.')